In [0]:
CREATE OR REPLACE TABLE workspace.default.invoice_headers AS
SELECT
  path,
  invoice_data:response.invoice_id.value::STRING AS invoice_id,
  invoice_data:response.invoice_date.value::STRING AS invoice_date,
  invoice_data:response.vendor_name.value::STRING AS vendor_name,
  invoice_data:response.vendor_address.value::STRING AS vendor_address,
  invoice_data:response.customer_name.value::STRING AS customer_name,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(invoice_data:response.subtotal.value::STRING, '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS subtotal,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(invoice_data:response.tax_amount.value::STRING, '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS tax_amount,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(invoice_data:response.total.value::STRING, '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS total,
  invoice_data:response.currency.value::STRING AS currency,
  invoice_data:response.due_date.value::STRING AS due_date,
  invoice_data:response.payment_terms.value::STRING AS payment_terms
FROM (
  SELECT
    path,
    ai_extract(
      ai_parse_document(content),
      '["invoice_id", "invoice_date", "vendor_name", "vendor_address", "customer_name", "subtotal", "tax_amount", "total", "currency", "due_date", "payment_terms"]'
    ) AS invoice_data
  FROM READ_FILES('/Volumes/workspace/default/invoices', format => 'binaryFile')
)

In [0]:
SELECT
  vendor_name,
  COUNT(*) AS invoice_count,
  SUM(subtotal) AS total_subtotal,
  SUM(tax_amount) AS total_tax,
  SUM(total) AS total_amount,
  currency
FROM workspace.default.invoices
GROUP BY vendor_name, currency
ORDER BY total_amount DESC

Databricks visualization. Run in Databricks to view.

In [0]:
-- Extract ALL line items from ALL invoices (handles multiple table formats)
WITH invoice_ids AS (
  SELECT
    path,
    invoice_data:response.invoice_id.value::STRING AS invoice_id
  FROM (
    SELECT
      path,
      ai_extract(
        ai_parse_document(content),
        '["invoice_id"]'
      ) AS invoice_data
    FROM READ_FILES('/Volumes/workspace/default/invoices', format => 'binaryFile')
  )
),
parsed AS (
  SELECT
    path,
    ai_parse_document(content) AS doc
  FROM READ_FILES('/Volumes/workspace/default/invoices', format => 'binaryFile')
),
elements_exploded AS (
  SELECT
    path,
    EXPLODE(from_json(doc:document.elements::STRING, 
      'array<struct<id:int,type:string,content:string>>')) AS element
  FROM parsed
),
table_rows AS (
  SELECT
    path,
    EXPLODE(
      SPLIT(
        REGEXP_EXTRACT(element.content, '<table>(.*?)</table>', 1),
        '<tr>'
      )
    ) AS row_html
  FROM elements_exploded
  WHERE element.type = 'table'
    AND (element.content LIKE '%nazwa towar%' OR element.content LIKE '%Nazwa towar%' OR element.content LIKE '%description%' OR element.content LIKE '%item%')
),
parsed_rows AS (
  SELECT
    path,
    row_html,
    REGEXP_EXTRACT_ALL(row_html, '<td>(.*?)</td>') AS cells
  FROM table_rows
  WHERE row_html LIKE '%<td>%'
    AND row_html NOT LIKE '%<th>%'
)
SELECT
  i.invoice_id,
  REGEXP_EXTRACT(p.path, 'invoices/([^/]+)$', 1) AS filename,
  -- Handle different table formats: item name may be in cells[0] or cells[1]
  CASE 
    WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[1]  -- If first cell is line number, item is in cells[1]
    ELSE cells[0]  -- Otherwise item is in cells[0]
  END AS item_name,
  -- Quantity position also differs: cells[2] or cells[3]
  TRY_CAST(
    CASE 
      WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[3]
      ELSE cells[2]
    END AS INT
  ) AS quantity,
  -- Unit position: cells[2] or cells[3]
  CASE 
    WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[2]
    ELSE cells[3]
  END AS unit,
  -- Unit price position: cells[4]
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[4], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS unit_price,
  -- Net value position: cells[5]
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[5], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS net_value,
  -- Gross value position: cells[8]
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[8], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS gross_value
FROM parsed_rows p
JOIN invoice_ids i ON p.path = i.path
WHERE SIZE(cells) >= 9
  AND (
    (cells[0] NOT LIKE '%razem%' AND cells[0] NOT LIKE '%Razem%' AND cells[0] NOT LIKE 'colspan%' AND cells[0] != '')
    OR 
    (cells[0] REGEXP '^[0-9]+$' AND cells[1] != '')
  )
ORDER BY i.invoice_id, item_name

In [0]:
-- Save all line items from all invoices to a table
CREATE OR REPLACE TABLE workspace.default.invoice_line_items AS
WITH invoice_ids AS (
  SELECT
    path,
    invoice_data:response.invoice_id.value::STRING AS invoice_id
  FROM (
    SELECT
      path,
      ai_extract(
        ai_parse_document(content),
        '["invoice_id"]'
      ) AS invoice_data
    FROM READ_FILES('/Volumes/workspace/default/invoices', format => 'binaryFile')
  )
),
parsed AS (
  SELECT
    path,
    ai_parse_document(content) AS doc
  FROM READ_FILES('/Volumes/workspace/default/invoices', format => 'binaryFile')
),
elements_exploded AS (
  SELECT
    path,
    EXPLODE(from_json(doc:document.elements::STRING, 
      'array<struct<id:int,type:string,content:string>>')) AS element
  FROM parsed
),
table_rows AS (
  SELECT
    path,
    EXPLODE(
      SPLIT(
        REGEXP_EXTRACT(element.content, '<table>(.*?)</table>', 1),
        '<tr>'
      )
    ) AS row_html
  FROM elements_exploded
  WHERE element.type = 'table'
    AND (element.content LIKE '%nazwa towar%' OR element.content LIKE '%Nazwa towar%' OR element.content LIKE '%description%' OR element.content LIKE '%item%')
),
parsed_rows AS (
  SELECT
    path,
    row_html,
    REGEXP_EXTRACT_ALL(row_html, '<td>(.*?)</td>') AS cells
  FROM table_rows
  WHERE row_html LIKE '%<td>%'
    AND row_html NOT LIKE '%<th>%'
)
SELECT
  i.invoice_id,
  REGEXP_EXTRACT(p.path, 'invoices/([^/]+)$', 1) AS filename,
  CASE 
    WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[1]
    ELSE cells[0]
  END AS item_name,
  TRY_CAST(
    CASE 
      WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[3]
      ELSE cells[2]
    END AS INT
  ) AS quantity,
  CASE 
    WHEN cells[0] REGEXP '^[0-9]+$' THEN cells[2]
    ELSE cells[3]
  END AS unit,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[4], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS unit_price,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[5], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS net_value,
  CAST(
    REPLACE(REPLACE(REGEXP_REPLACE(cells[8], '[^0-9,\\s]', ''), ' ', ''), ',', '.')
    AS DECIMAL(10,2)
  ) AS gross_value
FROM parsed_rows p
JOIN invoice_ids i ON p.path = i.path
WHERE SIZE(cells) >= 9
  AND (
    (cells[0] NOT LIKE '%razem%' AND cells[0] NOT LIKE '%Razem%' AND cells[0] NOT LIKE 'colspan%' AND cells[0] != '')
    OR 
    (cells[0] REGEXP '^[0-9]+$' AND cells[1] != '')
  )
ORDER BY i.invoice_id, item_name

In [0]:
-- Demonstrate EXPLODE using ARRAY() to create a synthetic array
-- This simulates what would happen with real line_items arrays

WITH invoice_with_array AS (
  SELECT 
    '4/2021' AS invoice_id,
    'RAFSOFT.NET' AS vendor_name,
    ARRAY(
      STRUCT('Service A' AS description, 2 AS quantity, 500.00 AS price),
      STRUCT('Service B' AS description, 1 AS quantity, 619.00 AS price)
    ) AS line_items
  UNION ALL
  SELECT 
    '298/2010' AS invoice_id,
    'Firma Demonstracyjna' AS vendor_name,
    ARRAY(
      STRUCT('Product X' AS description, 5 AS quantity, 400.00 AS price),
      STRUCT('Product Y' AS description, 3 AS quantity, 869.40 AS price),
      STRUCT('Product Z' AS description, 1 AS quantity, 730.60 AS price)
    ) AS line_items
)

SELECT 
  invoice_id,
  vendor_name,
  exploded_line.description,
  exploded_line.quantity,
  exploded_line.price,
  exploded_line.quantity * exploded_line.price AS line_total
FROM invoice_with_array
LATERAL VIEW EXPLODE(line_items) AS exploded_line
ORDER BY invoice_id, exploded_line.description